In [ ]:
!pip -q install captum
# Importa un modello DenseNet da torchvision

# 1. Setup del modello e dataset
# Carichiamo DenseNet pre-addestrato
model = densenet121(weights=DenseNet121_Weights.DEFAULT)

# Trasformazioni per il dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # DenseNet si aspetta input 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225])
])

# Carica il dataset Faces in the Wild
dataset = torchvision.datasets.LFWPeople(root='./data',
                                       split='train',
                                       transform=transform,
                                       download=True)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Freeziamo tutti i parametri della rete
for param in model.parameters():
    param.requires_grad = False

# Modifichiamo l'ultimo layer per il numero corretto di classi
num_classes = len(dataset.class_to_idx)
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, num_classes)

print(f"Numero di classi nel dataset: {num_classes}")

# Training loop
num_epochs = 1
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 10 == 9:    # stampa ogni 10 mini-batch
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 10:.3f}')
            running_loss = 0.0
            break

print('Fine-tuning completato')

# Mostra spiegazioni con Captum Per il viso numero 25 usando Occlusion, Saliency e Integrated Gradients


